In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch5. LSTM</font>
- 영화평 감성 분석

# 1. pkg 불러오기

In [ ]:
import numpy as np
from time import time
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
import pandas as pd

# 2. 하이퍼 파라미터 설정
- 파라미터 설정에 따라 score와 처리 속도 등이 달라짐

In [ ]:
MY_WORDS = 10000 # imdb 데이터셋의 단어 수
MY_LENGTH = 80 # 영화평 단어 수 80개까지 독립 변수 (추천 : 200)
MY_EMBED = 32 # 임베딩 레이어의 출력 차원 (추천 : 128)
MY_HIDDEN = 64 # LSTM의 units 수 (추천 : 256)

MY_EPOCH = 10 # 반복 학습 수
MY_BATCH = 200 # 매번 가져오는 데이터 수

SKIP_TOP = 40 # 불용어 인덱스 설정 (빈도수 상위 40개 제외 : the, a, is 등)

# 3. 데이터 불러오기

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=MY_WORDS,
    skip_top=SKIP_TOP
)

In [ ]:
print('학습용 데이터 shape :', X_train.shape, y_train.shape)
print('학습용 입력 데이터 샘플 :',X_train[0][:100], '-', type(X_train[0]), '-', len(X_train[0]))
print('학습용 출력 데이터 샘플 (0 - 부정 / 1 - 긍정) :',y_train[0], '-', type(y_train[0]))

print('테스트용 데이터 shape :', X_test.shape, y_test.shape)
print('테스트용 입력 데이터 샘플 :',X_train[0][:100], '-', type(X_test[0]), '-', len(X_test[0]))
print('테스트용 출력 데이터 샘플 (0 - 부정 / 1 - 긍정) :',y_test[0], '-', type(y_test[0]))

In [ ]:
# X_train의 길이 및 평균 길이
print([len(x) for x in X_train[:10]])
np.array([len(x) for x in X_train]).mean()

In [ ]:
# 출력 데이터 샘플 균형 확인
print('학습용 데이터의 "긍정" 개수 :', y_train.sum())
print('테스트용 데이터의 "긍정" 개수 :', y_test.sum())

In [ ]:
pd.Series(y_train).value_counts()

# 4. 문자 단어 → 정수 변환

In [ ]:
word_to_id = imdb.get_word_index() # dict {'word':idx}
print(word_to_id['movie'])
print(word_to_id['film'])
print(word_to_id['the'])

In [ ]:
# {'word':id} → {id:'word'} 형태 변경
id_to_word = {}
for word, id in word_to_id.items():
    id_to_word[id] = word
    
id_to_word[29]

In [ ]:
# 감성 분석에 중요도가 떨어지는 단어
print([id_to_word.get(i) for i in range(1,40)])

In [ ]:
msg = 'What a wonderful movie'
msg = msg.lower().split()
# idx 1 : 리뷰의 시작 부분에 무조건 추가되는 인덱스
# idx 2 : 잘려진 문자를 잘못 읽어온 경우 추가되는 인덱스
# idx 3 : padding 처리를 위해 추가되는 인덱스

# idx 1,2,3과 겹치는 것을 피하기 위해 단어들의 인덱스에 3 추가
data = [1]+ [word_to_id.get(m, -1) + 3 for m in msg]

# 단어들의 인덱스 검색을 위해 단어들의 인덱스에서 3 감소
data_msg = ' '.join([id_to_word.get(d-3, '??') for d in data])


print('후기 내용 원본 :', msg)
print('인코딩된 데이터 :', data)
print('data로 추정한 msg :', data_msg)

# 5. 숫자화된 영화평 → 자연어 영화평 변환 함수

In [ ]:
def decoding(review_num):
    "숫자화된 영화평을 자연어 영화평으로 바꿔 return"
    decoded = [id_to_word.get(d-3, '??') for d in review_num]
    return ' '.join(decoded)
decoding(X_train[0])

# 6. 영화평 데이터 길이 출력 함수

In [ ]:
def show_length(X_train=X_train):
    print('영화평 데이터 5개 단어 길이')
    for i in range(5):
        print(f'{i}번째 길이 : {len(X_train[i])}')

In [ ]:
show_length()

In [ ]:
# 영화평 단어 길이 최댓값과 최솟값
max(len(x) for x in X_train), min(len(x) for x in X_train)

# 7. 모든 영화평 길이 통일
- 기준 : MY_LENGTH
- pad_sequences 활용

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
X_train = pad_sequences(X_train,
                        maxlen=MY_LENGTH, # 허용 최대 길이 설정
                        truncating='post', # MY_LENGTH를 넘어갈 경우, 제거할 위치 설정
                        padding='pre', # MY_LENGTH보다 부족할 경우, 부족한 공간에 '0' 추가(zero padding)
                        )

X_test = pad_sequences(X_test,
                       maxlen=MY_LENGTH,
                       truncating='post',
                       padding='pre',
                       )
show_length(X_train)
show_length(X_test)

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

# 8. 모델 생성 구현

- RNN의 한계점
     * 장기 의존성 문제(시퀀스 길이가 길어질수록 과거의 주요 정보가 후반부까지 전달되지 못하고 희석됨), 기울기 소실 및 폭주
- RNN의 개선 모델
    1. LSTM
    2. GRU
    
    
## LSTM
- 셀 상태 ($C_t$ - 메인 장부) : 과거부터 현재까지의 핵심 정보가 손실 없이 흐르는 고속도로 역할
- 은닉 상태 ($h_t$ - 요약 보고서) : 현재 시점에서 외부로 출력되거나 다음 단계로 전달되는 요약 정보
- 4개 연산 레이어 (검토 부서)
    * 망각 게이트 ($f_t$) : 옛날 기록 중 필요 없는 것을 얼마나 지울지 결정 ($0 \sim 1$)
    * 입력 게이트 선택 ($i_t$) : 오늘 들어온 새 정보 중 얼마만큼 반영할지 선택 ($0 \sim 1$)
    * 입력 게이트 후보 ($\tilde{C}_t$) : 과거 맥락과 현재 데이터를 결합해 재해석한 새 정보 후보
    * 출력 게이트 ($o_t$) : 업데이트된 장부에서 얼마나 외부에 보여줄지 선택 ($0 \sim 1$)
    
 
 - 동작 방식 및 순서
     * [1. 입력 결합] ──► [2. 지우기] ──► [3. 새 정보 채우기] ──► [4. 장부 갱신] ──► [5. 요약 출력]

     1. 입력 결합: 현재 시점 단어($x_t$)와 이전 요약본($h_{t-1}$)을 하나로 병합
     2. 지우기 (망각) : 과거 장부($C_{t-1}$)에 망각 게이트 비율($f_t$)을 곱해 불필요한 기억 제거
     3. 새 정보 채우기 (입력) : 입력 선택 비율($i_t$)과 새 후보 정보($\tilde{C}_t$)를 곱해 장부에 추가
     4. 장부 갱신 ($C_t$) : 지우고 더해진 최종 최신 장부($C_t$) 완성
     5. 요약 출력 ($h_t$) : 최신 장부($C_t$)에 출력 게이트 비율($o_t$)을 곱해 최종 출력값($h_t$) 확정

In [ ]:
model = Sequential(name='seq')

model.add(Embedding(
    input_dim=MY_WORDS,
    output_dim=MY_EMBED,
    input_length=MY_LENGTH))

model.add(LSTM(
    units=MY_HIDDEN,
    dropout=0.3,                 # 각 step마다 입력 데이터에서 LSTM 셀로 들어가는 연결의 30%를 무작위로 0으로
    recurrent_dropout=0.3,       # 이전 은닉 상태 순환 연결의 30% 무작위 차단
    kernel_regularizer=l2(0.001),# L2 가중치 규제 (입력 가중치 폭주 방지)
    kernel_initializer='he_normal',       # 입력 가중치(W_x) 초기화
    recurrent_initializer='orthogonal'    # 순환 가중치(W_h) 직교 초기화 (기울기 소실 방지)
))

model.add(Dense(units=1, activation='sigmoid'))

model.summary()

# 9. 학습과정 설정 및 학습

In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])

In [ ]:
begin = time()

earlyStopping = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    mode='min', 
    restore_best_weights=True, # 멈출 경우, monitor가 확인 중인 값이 가장 낮았던 시점의 가중치로 복원
    verbose=1
)

hist = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=MY_EPOCH,
    batch_size=MY_BATCH,
    verbose=2,
    callbacks=[earlyStopping]
)

end = time()

print(f'총 학습시간 : {round(end - begin, 2)}초')

# 10. 모델 평가

In [ ]:
import matplotlib.pyplot as plt

fig, loss_ax = plt.subplots(figsize=(10,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
loss_ax.plot(hist.history['val_loss'], 'r', label='val_loss')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss', rotation=0)

acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['acc'], 'g', label='train_accuracy')
acc_ax.plot(hist.history['val_acc'], 'b', label='val_accuracy')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.6), ncol=1)
acc_ax.legend(loc='center right', bbox_to_anchor=(1.5, 0.8), ncol=1)
plt.show()

In [ ]:
loss, acc = model.evaluate(X_test, y_test)

In [ ]:
# 혼돈 행렬을 위한 예측값
pred = model.predict(X_test, batch_size=MY_BATCH)
y_hat = (pred>0.5).astype(int)

In [ ]:
confusion_matrix(y_test, y_hat)

In [ ]:
pd.crosstab(y_test, y_hat.reshape(-1))

In [ ]:
print('recall :', recall_score(y_test, y_hat))
print('precision :', precision_score(y_test, y_hat))
print('f1-score :', f1_score(y_test, y_hat))

# 11. 모델 사용하기

In [ ]:
review = '''I was just fine with everything. Actually, I think it would have been fine even if it wasn’t IMAX
The overwhelming impact of the 4:3 aspect ratio shone through in some scenes, but
Overall, "Cinematography" is just cool
And the casting that didn’t fit at all didn’t cause any problems for either my wife and me
The moment when the first godic being Odysseus encounters after leaving Troy
The audience (at least my wife and I) already feel like we’ve fallen right in the middle of the stage in the myth, so
No matter who I saw—Asian actors, Indian actors, or Black actors—it didn’t bother me'''.lower()
import re
review = re.sub('[^a-zA-Z\s]', ' ', review).split()
review = [1] + [word_to_id.get(word, -1) + 3 if (word_to_id.get(word, -1) + 3)<10000 else 2 for word in review]
print(review)

In [ ]:
input_data = pad_sequences(
    [review],
    maxlen=MY_LENGTH,
    truncating='post',
    padding='pre',
    )
input_data.shape

In [ ]:
(model.predict(input_data)>0.5).astype('int')

In [ ]:
review = '''They say the higher the expectations, the bigger the disappointment, 
and that completely sums up this movie. While the fresh premise and flashy visuals 
initially caught my eye, it stopped right there. By the middle, the plot completely 
lost its logic, making the characters' actions impossible to understand.
The lines were so cheesy and cliché that it felt like a waste of the actors' solid 
performances. Dragging scenes had me constantly checking my watch, and even the 
climax left me feeling flat. The story got so tangled that it was hard to guess 
what message the director was try'''.lower()
review = [1] + [word_to_id.get(r, -1)+3 for r in review]
print(review)
input_data = pad_sequences([review],
               maxlen=MY_LENGTH,
               truncating='post', # 80단어 이상일 경우 어디를 짜를지 여부
               padding='pre', # 80단어 미만일 경우 앞에 zero를 붙임
               )
(model.predict(input_data)>0.5).astype('int')